In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("new app").getOrCreate()

In [0]:
df1 = spark.read.csv("file.csv",header=True,inferSchema=True)

In [0]:
data = [
(101,'Naina','2026-04-03'),
(102,'Anu','2026-07-03'),
(105,'Yashika','2026-04-03'),
(103,'Akshya','2026-08-06'),
(104,'Saksham','2026-02-09'),
(105,'Yashika','2026-06-03'),
(103,'Akshya','2026-01-01')
]

col = ["customer_id","name","updated_date"]
df = spark.createDataFrame(data,col)
df.display()

In [0]:
df.printSchema()

In [0]:
df.select("name").show()

In [0]:
#In df there are duplicate records and I want unique record having lastest updated_date
from pyspark.sql import Window
from pyspark.sql.functions import *
window_spec = Window.partitionBy("customer_id").orderBy(col("updated_date").desc())
df_result = df.withColumn("rnk", row_number().over(window_spec)).filter(col("rnk")==1).drop("rnk")
#drop rnk means after doing window fun we are removing rnk col as we don't want it to be printed
df_result.show()

# → column par function/method .desc() apply karna hai, isliye col() use karna convenient hai.esliea humne partition by krte time col use nhi kra but orderby ke time pr kra hai

In [0]:
#Find highest salary per department
from pyspark.sql import Window
from pyspark.sql.functions import *
window_spec = Window.partitionBy("department_name").orderBy(col("salary").desc())
df_result = df.withColumn("rnk",dense_rank().over(window_spec)).filter(col("rnk")==1).drop("rnk")
df_result.show()

In [0]:
# Find month over month revenue Growth(it is basically a cumulative sum que) We are having col as customer_id, month, revenue

from pyspark.sql import Window
from pyspark.sql.functions import *
window_spec = Window.partitionBy("customer_id").orderBy("month")
df_result = df.withColumn("previous_month_revenue",lag("revenue").over(window_spec).withColumn("revenue_diffrence",col("revenue")-col("previous_month_revenue")))
df_result.show()

In [0]:
# Que. Optimize a join, We are having 2 df
# Trasaction_df having col as txn_id, cust_id, country_code, amount
# Country_df having col as country_code, country_name

#applyting normal join
display(transaction_df.join(country_df,'country_code'))

#optimizing it using broadcast join, we will get the same result here as well
display(transaction_df.join(broadcast(country_df),'country_code'))

In [0]:
#incremental Load
from pyspark.sql.functions import *

orders_data = [
    (1,"Created","2024-01-01"),
    (2,"Shipped","2024-01-05"),
    (3,"Delivered","2024-01-10"),
    (4,"Cancelled","2024-01-15")
]
last_processed_date = '2024-01-10'
orders_df = spark.createDataFrame(orders_data,['order_id','status','updated_at'])
display(orders_df)

#I want last success3ful run/last process date and all these kind of data.I want to load new data in this record

incremental_df = orders_df.filter(col("updated_at")>last_processed_date)
display(incremental_df)

In [0]:
customer_data = [
    (
        101,
        "Anurag",
        [
            {"order_id": 1, "product": "Laptop", "amount": 50000},
            {"order_id": 2, "product": "Mouse", "amount": 1000}
        ]
    ),
    (
        102,
        "Stuti",
        [
            {"order_id": 3, "product": "Keyboard", "amount": 2500}
        ]
    ),
    (
        103,
        "Rahul",
        []
    )
]

In [0]:
# Que. If I’m getting data from an API , JSON kind of data then how would i flatten that kind of data

from pyspark.sql import Window
from pyspark.sql.types import *
from pyspark.sql.functions import *

order_schema = ArrayType(
    StructType([
        StructField("order_id",IntegerType(),True), # True defines nullable
        StructField("product",StringType(),True),
        StructField("amount",IntegerType(),True) 
    ])
)
customer_schema = StructType([
        StructField("customer_id",IntegerType(),True), # True defines nullable
        StructField("customer_name",StringType(),True),
        StructField("orders",order_schema,True) 
    ])


In [0]:
#Display data in dataframe
customer_df = spark.createDataFrame(customer_data,customer_schema)
display(customer_df)

In [0]:
#Explode the Array
explode_df = customer_df.withColumn("order",explode(col("orders")))
display(explode_df)

In [0]:
# Union Data with different column orders.
# You are getting a daily file where the columns are same but the order of the columns are different.

day1_data = [
    (1,'Anurag', 50000),
    (2,'Stuti', 60000)
]
day2_data = [
    (70000,3,'David'),
    (80000,4,'Sophia')
]
day1_data_df = spark.createDataFrame(day1_data,["emp_id","emp_name","salary"])
day2_data_df = spark.createDataFrame(day2_data,["salary","emp_id","emp_name"])
display(day1_data_df)
display(day2_data_df)

#For this kind of problem there is a function known as UnionByName
union_df = day1_data_df.unionByName(day2_data_df)
display(union_df)

# Follow-up Let say ab ek new DF aa gaya having an extra col that is not present in previous DF
day3_data = [
    (5, "Vaishnav", 90000,"IT")
]
day3_data_df = spark.createDataFrame(day3_data,["emp_id","emp_name","salary","department"])
display(day3_data_df)

union_df1 = day3_data_df.unionByName(union_df, allowMissingColumns=True)
display(union_df1)
